# 3. Did the simplified process help, and at what cost?

We compare customers by original assignment. Negative changes improve the capped waiting-time endpoint. Rate differences are shown in percentage points; costs are USD per assigned customer.

In [1]:
from pathlib import Path
import sys, json
import numpy as np
import polars as pl
import matplotlib.pyplot as plt
from IPython.display import display, Markdown
ROOT = Path.cwd()
if not (ROOT / 'config').exists():
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT / 'src'))
from support_policy.common import config
mart = pl.read_parquet(ROOT / 'data/marts/experiment_mart.parquet')
plt.rcParams.update({'figure.figsize': (9, 4), 'axes.spines.top': False, 'axes.spines.right': False})
pl.Config.set_tbl_rows(20)
print(f'{mart.height:,} customers; synthetic data only')


100,000 customers; synthetic data only


## Unadjusted intention-to-treat estimates
Each customer contributes once. We use Welch intervals for mean differences; binary means are risks. Survey ratings remain a respondent-only descriptive exception.

In [2]:
from support_policy.estimate_effects import estimate
effects=estimate(mart)
display(effects.select('metric','n_control','n_treatment','control_mean','treatment_mean','effect','ci_low','ci_high','p_value'))


metric,n_control,n_treatment,control_mean,treatment_mean,effect,ci_low,ci_high,p_value
str,i64,i64,f64,f64,f64,f64,f64,f64
"""unresolved_hours_7d""",50180,49820,44.518952,39.286769,-5.232183,-5.786891,-4.677475,3.5087e-76
"""repeat_contact_7d""",50180,49820,0.100219,0.119892,0.019672,0.015794,0.023551,2.7942e-23
"""escalated_7d""",50180,49820,0.054384,0.05285,-0.001534,-0.004326,0.001258,0.281608
"""refund_usd_7d""",50180,49820,3.690014,4.125754,0.43574,0.379552,0.491927,4.0580e-52
"""resolved_7d""",50180,49820,0.925528,0.939542,0.014014,0.010907,0.017122,9.8052e-19
"""csat""",24696,22336,4.003766,3.908623,-0.095143,-0.109769,-0.080517,3.5820e-37
"""survey_responded""",50180,49820,0.492148,0.448334,-0.043814,-0.049995,-0.037633,7.6534e-44


## Explain the primary outcome in ordinary language

In [3]:
r=effects.filter(pl.col('metric')=='unresolved_hours_7d').row(0,named=True)
print(f"The new policy changed capped time to first resolution by {r['effect']:.2f} hours.")
print(f"The 95% interval is {r['ci_low']:.2f} to {r['ci_high']:.2f} hours.")
print('This is customer waiting time. It is not measured agent labour saved.')


The new policy changed capped time to first resolution by -5.23 hours.
The 95% interval is -5.79 to -4.68 hours.
This is customer waiting time. It is not measured agent labour saved.


## Operational guardrails
The decision uses simultaneous one-sided upper bounds. The point estimate alone cannot establish that harm is acceptably small.

In [4]:
guards=effects.filter(pl.col('guardrail_limit').is_not_null())
display(guards.select('metric','effect','ci_low','ci_high','guardrail_upper','guardrail_limit','guardrail_pass'))
repeat=effects.filter(pl.col('metric')=='repeat_contact_7d').row(0,named=True)
print(f"Repeat contacts changed by {100*repeat['effect']:.2f} percentage points.")
print(f"Simultaneous upper bound: {100*repeat['guardrail_upper']:.2f} points; allowed limit: {100*repeat['guardrail_limit']:.2f} points.")


Repeat contacts changed by 1.97 percentage points.
Simultaneous upper bound: 2.39 points; allowed limit: 1.00 points.


metric,effect,ci_low,ci_high,guardrail_upper,guardrail_limit,guardrail_pass
str,f64,f64,f64,f64,f64,bool
"""repeat_contact_7d""",0.019672,0.015794,0.023551,0.023883,0.01,false
"""escalated_7d""",-0.001534,-0.004326,0.001258,0.001498,0.005,true
"""refund_usd_7d""",0.43574,0.379552,0.491927,0.496745,0.5,true


## Baseline-adjusted sensitivity
This model adjusts only for pre-assignment attributes. HC3 uncertainty allows heteroskedasticity. The unadjusted result remains the planned primary analysis.

In [5]:
display(pl.read_csv(ROOT/'outputs/tables/adjusted_primary.csv'))

metric,method,effect,ci_low,ci_high
str,str,f64,f64,f64
"""unresolved_hours_7d""","""OLS baseline adjusted; HC3""",-5.239638,-5.785938,-4.693339


## Satisfaction and the missing denominator
A difference among responders does not estimate the effect for all assigned customers when treatment changes response.

In [6]:
display(effects.filter(pl.col('metric').is_in(['csat','survey_responded'])).select('metric','n_control','n_treatment','control_mean','treatment_mean','effect','ci_low','ci_high'))

metric,n_control,n_treatment,control_mean,treatment_mean,effect,ci_low,ci_high
str,i64,i64,f64,f64,f64,f64,f64
"""csat""",24696,22336,4.003766,3.908623,-0.095143,-0.109769,-0.080517
"""survey_responded""",50180,49820,0.492148,0.448334,-0.043814,-0.049995,-0.037633


## Check against the simulation design, after analysis
Only this diagnostic accesses counterfactual aggregate truth. It was not used to select models or make the policy decision. One interval can miss truth by chance.

In [7]:
display(pl.read_csv(ROOT/'outputs/tables/simulation_recovery.csv'))
display(json.loads((ROOT/'outputs/estimator_calibration.json').read_text()))


metric,finite_sample_true_effect,estimate,estimation_error,truth_in_95_interval
str,f64,f64,f64,bool
"""unresolved_hours_7d""",-5.325049,-5.232183,0.092866,true
"""repeat_contact_7d""",0.02361,0.019672,-0.003938,false
"""escalated_7d""",0.00099,-0.001534,-0.002524,true
"""refund_usd_7d""",0.4019795,0.43574,0.03376,true


{'replications': 200,
 'null_ci_coverage': 0.935,
 'null_false_positive_rate': 0.065,
 'note': 'Diagnostic on normal synthetic outcomes; not a guarantee for every distribution.'}

## Read the trade-off, not only the p-value
A large sample can make small effects statistically clear. The rollout gate therefore combines a minimum practical benefit with predeclared harm limits. Notebook 4 applies that gate and examines follow-up questions.